# nanochat.cpp end to end on Kaggle (2x T4)

Build the CUDA library for the T4 (`sm_75`), prepare ClimbMix and the
`NCTOKEN1` tokenizer, run the Python API checks, and train a small model in one
process.

The runtime uses one T4. The C++ workflow has no distributed mode
(`docs/host-portability.md`). The Python surface is `docs/python.md`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("HF_TOKEN", "GH_TOKEN", "OPENCODE_API_KEY"):
    try:
        os.environ[name] = secrets.get_secret(name)
    except Exception:
        print("no secret:", name)

REPO = Path("/kaggle/working/nanochat.cpp")
BASE = Path("/kaggle/working/nanochat-cache")
HOME = Path(os.environ["HOME"])

os.environ.update(
    NANOCHAT_REPO="ming6ao/nanochat.cpp",
    NANOCHAT_SANDBOX_BACKEND="none",
    NANOCHAT_CPP_BACKEND="cuda",
    NANOCHAT_CPP_PRECISION="fp16",
    NANOCHAT_CUDA_ARCH="sm_75",
    NANOCHAT_CPP_CACHE=str(BASE / "build"),
    CUDA_HOME="/usr/local/cuda",
    GIT_AUTHOR_NAME="Kaggle Notebook",
    GIT_AUTHOR_EMAIL="gaomingduke@gmail.com",
)
os.environ["PATH"] = os.pathsep.join([
    str(HOME / ".local/bin"),
    str(HOME / ".local/node/bin"),
    "/usr/local/cuda/bin",
    os.environ["PATH"],
])

# The interactive terminal does not inherit os.environ.
keys = ("GH_TOKEN", "OPENCODE_API_KEY", "NANOCHAT_REPO",
        "NANOCHAT_SANDBOX_BACKEND", "NANOCHAT_CPP_BACKEND",
        "NANOCHAT_CPP_PRECISION", "NANOCHAT_CUDA_ARCH", "CUDA_HOME",
        "NANOCHAT_CPP_CACHE", "PATH")
with open(HOME / ".nanochat.env", "w") as handle:
    for key in keys:
        handle.write(f'export {key}="{os.environ[key]}"\n')

print("environment ready:", REPO)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total",
                      "--format=csv,noheader"], text=True,
                     capture_output=True).stdout.strip())


In [ ]:
if not REPO.is_dir():
    subprocess.run(["git", "clone",
                    f"https://github.com/{os.environ['NANOCHAT_REPO']}",
                    str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
subprocess.run(["tools/kaggle/bootstrap.sh"], cwd=str(REPO), check=True)

revision = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short",
                           "HEAD"], text=True, capture_output=True)
print("revision:", revision.stdout.strip())


In [ ]:
sys.path.insert(0, str(REPO / "python"))

import nanochat_cpp as nc

# Build the shared library for the T4 and pin the path before the first load.
os.environ["NANOCHAT_CPP_LIB"] = str(nc.build())
print("CUDA library:", os.environ["NANOCHAT_CPP_LIB"])


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "huggingface_hub"], check=True)

from huggingface_hub import snapshot_download

DATA = BASE / "base_data_climbmix"
TRAIN_PATTERN = "shard_0000*.parquet"   # shards 00000 to 00009
VAL_SHARD = "shard_06542.parquet"       # the reference validation split

if not (DATA / VAL_SHARD).is_file():
    snapshot_download(repo_id="karpathy/climbmix-400b-shuffle",
                      repo_type="dataset", local_dir=str(DATA),
                      allow_patterns=[TRAIN_PATTERN, VAL_SHARD],
                      token=os.environ.get("HF_TOKEN"))

shards = sorted(DATA.glob("shard_*.parquet"))
print(len(shards), "shards; validation:", (DATA / VAL_SHARD).name)


In [ ]:
artifact = BASE / "tokenizer" / "tokenizer.nctoken"

if not artifact.is_file():
    artifact.parent.mkdir(parents=True, exist_ok=True)
    # The orchestration layer calls tools/nanochat for the build and the run.
    nc.toolchain.build(["//src/tokenizer:tok_train_main"])
    nc.toolchain.run("t0-cpu", [
        "bazel-bin/src/tokenizer/tok_train_main",
        "--parquet", f"{DATA}/shard_*.parquet",
        "--vocab-size", "32768",
        "--out", str(artifact),
    ])

print("tokenizer:", artifact, artifact.stat().st_size, "bytes")


## Checks

The checks fail the notebook on the first error. They cover the tokenizer, the
planning layer, a device model step, the chat prompt renderer, and the
toolchain report.


In [ ]:
def check(condition, message):
    if not condition:
        raise AssertionError(message)
    print("ok:", message)


tokenizer = nc.Tokenizer.load(str(artifact))

# The tokenizer round trip.
ids = tokenizer.encode("The capital of France is")
check(bool(ids), "encode returns a token list")
check(tokenizer.decode(ids) == "The capital of France is",
      "decode inverts encode")

# The planning layer.
plan = nc.plan.compute_plan(
    depth=4, vocab_size=tokenizer.vocab_size, seq_len=128,
    device_batch_size=2, total_batch_size=256, num_iterations=2)
counts = nc.plan.params(plan.config)
check(counts.total == counts.embeddings + counts.transformer_matrices
      + counts.lm_head + counts.scalars, "plan total is the group sum")
check(counts.scaling_params == counts.transformer_matrices + counts.lm_head,
      "scaling params sum the matrices and the head")
check(plan.grad_accum == 1, "the micro batch fills the total batch")

# A tiny model on the device.
tiny = nc.Model(nc.Config(depth=2, seq_len=64,
                          vocab_size=tokenizer.vocab_size),
                device="cuda", seed=1)
check(tiny.device.startswith("cuda"), "the model runs on cuda")
tokens = list(range(64))
loss = tiny.forward_loss(tokens, tokens[1:] + [0], batch=1, seq=64)
tiny.zero_grad()
tiny.backward()
check(loss == loss, "the forward loss is not NaN")

# The chat prompt renderer.
conversation = {"messages": [
    {"role": "user", "content": "Pick one. A or B."},
    {"role": "assistant", "content": "A"}]}
chat_evaluator = nc.chat.ChatEvaluator(tiny, tokenizer=tokenizer)
prompt = chat_evaluator.renderer.render_for_completion(conversation)
check(prompt[0] == tokenizer.get_bos_token_id(), "the prompt starts with bos")
check(prompt[-1] == tokenizer.encode_special("<|assistant_start|>"),
      "the prompt ends with the assistant start token")

# The toolchain report.
check(bool(nc.toolchain.doctor().repo), "doctor reports the repository")

print("all notebook checks passed")


## Train, evaluate, and generate

The plan sets the model shape, the horizon, and the scaled rates. The trainer
uses one micro batch, so the gradient accumulation is one.


In [ ]:
import time


def lr_multiplier(step, num_iterations, warmup_steps, warmdown_ratio,
                  final_lr_frac):
    """Mirror nanochat::Scheduler::LrMultiplier (src/optim.cc, 1-based)."""
    it = step - 1
    warmdown = int(warmdown_ratio * num_iterations + 0.5)  # std::lround
    if warmup_steps > 0 and it < warmup_steps:
        return (it + 1) / warmup_steps
    if warmdown <= 0 or it <= num_iterations - warmdown:
        return 1.0
    progress = (num_iterations - it) / warmdown
    return progress + (1.0 - progress) * final_lr_frac


# Peak FP16 tensor-core throughput for one T4. The C++ MFU table has no T4
# entry, so the notebook supplies the number.
PEAK_FLOPS_T4 = 65e12

plan = nc.plan.compute_plan(
    depth=8, vocab_size=tokenizer.vocab_size, seq_len=512,
    device_batch_size=8, total_batch_size=4096, num_iterations=50)
model = nc.Model(plan.config, device="cuda", seed=42)
print("backend:", model.device, "scaling params:", f"{plan.scaling_params:,}")

train_data = nc.TokenData(
    parquet=f"{DATA}/shard_0000*.parquet", tokenizer=tokenizer,
    seq_len=plan.config.seq_len, batch=plan.device_batch_size, threads=4)
optimizer = nc.Optimizer(model, num_iterations=plan.num_iterations,
                         **plan.optimizer, **plan.scheduler)
trainer = nc.Trainer(model, train_data, num_iterations=plan.num_iterations,
                     optimizer=optimizer)

# One optimizer step per iteration, so a step consumes one micro batch.
tokens_per_step = plan.device_batch_size * plan.config.seq_len
schedule = plan.scheduler
history = {"step": [], "loss": [], "grad_norm": [], "lr_mult": [],
           "matrix_lr": [], "tokens_per_sec": [], "mfu": []}

previous = None
for step, loss in trainer:
    now = time.perf_counter()
    seconds = None if previous is None else now - previous
    previous = now
    mult = lr_multiplier(step, plan.num_iterations,
                         schedule["warmup_steps"], schedule["warmdown_ratio"],
                         schedule["final_lr_frac"])
    norm = optimizer.grad_norm()
    history["step"].append(step)
    history["loss"].append(loss)
    history["grad_norm"].append(norm)
    history["lr_mult"].append(mult)
    history["matrix_lr"].append(plan.optimizer["matrix_lr"] * mult)
    if seconds is not None:
        rate = tokens_per_step / seconds
        history["tokens_per_sec"].append(rate)
        history["mfu"].append(rate * plan.flops_per_token / PEAK_FLOPS_T4)
    else:
        history["tokens_per_sec"].append(float("nan"))
        history["mfu"].append(float("nan"))
    if step == 1 or step % 10 == 0 or step == plan.num_iterations:
        print(f"step {step:3d}/{plan.num_iterations}  loss {loss:.4f}  "
              f"grad_norm {norm:.3f}  lr_mult {mult:.3f}")

losses = history["loss"]

checkpoint = Path("/kaggle/working/model.nchkpt01")
model.save(str(checkpoint))

report = nc.evaluate(
    model, parquet=str(DATA / VAL_SHARD), tokenizer=tokenizer,
    tokens=plan.device_batch_size * plan.config.seq_len * 4,
    batch=plan.device_batch_size, seed=43)
print(f"checkpoint: {checkpoint.stat().st_size:,} bytes")
print(f"bits per byte: {report.bpb:.4f}")

prompt = tokenizer.encode("The capital of France is")
row = model.generate(prompt, max_tokens=16, temperature=0.0)[0]
generated = [token for token, mask in zip(row.tokens, row.mask) if mask]
print("completion:", tokenizer.decode(generated))

# The after-training checks.
check(losses[-1] < losses[0], "the training loss decreased")
reloaded = nc.Model(plan.config, device="cuda", seed=0)
reloaded.load(str(checkpoint))
batch_tokens = list(range(plan.config.seq_len))
reloaded_loss = reloaded.forward_loss(batch_tokens, batch_tokens[1:] + [0],
                                      batch=1, seq=plan.config.seq_len)
check(reloaded_loss == reloaded_loss,
      "the reloaded model scores a finite loss")
print("post-training checks passed")


## Training metrics

The cell below plots the diagnostics from the run above. The four panels are:

1. The cross-entropy loss, with a centered mean over five steps.
2. The global gradient norm before clipping.
3. The matrix learning rate from the schedule.
4. The throughput in tokens per second, and the model FLOPs utilization (MFU).

The cell writes the figure to `/kaggle/working/training_metrics.png`. The
notebook saves that file as a Kaggle output.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def running_mean(values, window):
    """A centered mean over `window` samples that keeps the ends."""
    values = np.asarray(values, dtype=float)
    if window <= 1 or values.size < window:
        return values
    pad = window // 2
    padded = np.pad(values, pad, mode="edge")
    kernel = np.ones(window) / window
    return np.convolve(padded, kernel, mode="valid")[:values.size]


steps = np.asarray(history["step"], dtype=int)
loss = np.asarray(history["loss"], dtype=float)
grad_norm = np.asarray(history["grad_norm"], dtype=float)
matrix_lr = np.asarray(history["matrix_lr"], dtype=float)
tokens_per_sec = np.asarray(history["tokens_per_sec"], dtype=float)
mfu = np.asarray(history["mfu"], dtype=float)
# The floor keeps the log scale finite when a norm is zero.
grad_norm_plot = np.maximum(grad_norm, 1e-6)

figure, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

# The cross-entropy loss and its mean over five steps.
axis = axes[0, 0]
axis.plot(steps, loss, linewidth=1.0, alpha=0.4, label="loss")
axis.plot(steps, running_mean(loss, 5), linewidth=2.0,
          label="loss (mean of 5)")
axis.text(0.97, 0.95, f"validation bits per byte {report.bpb:.3f}",
          transform=axis.transAxes, ha="right", va="top", fontsize=9)
axis.set_title("Training loss")
axis.set_xlabel("step")
axis.set_ylabel("cross-entropy loss")
axis.grid(True, alpha=0.3)
axis.legend()

# The global gradient norm before clipping.
axis = axes[0, 1]
axis.plot(steps, grad_norm_plot, linewidth=1.5, color="tab:red")
axis.set_title("Gradient norm")
axis.set_xlabel("step")
axis.set_ylabel("global L2 norm")
axis.set_yscale("log")
axis.grid(True, alpha=0.3)

# The scheduled matrix learning rate.
axis = axes[1, 0]
axis.plot(steps, matrix_lr, linewidth=1.5, color="tab:green")
axis.set_title("Matrix learning rate")
axis.set_xlabel("step")
axis.set_ylabel("learning rate")
axis.grid(True, alpha=0.3)

# The throughput and the model FLOPs utilization.
axis = axes[1, 1]
axis.plot(steps, tokens_per_sec, linewidth=1.5, color="tab:blue")
axis.set_title("Throughput")
axis.set_xlabel("step")
axis.set_ylabel("tokens / second")
axis.grid(True, alpha=0.3)
twin = axis.twinx()
twin.plot(steps, 100.0 * mfu, linewidth=1.2, color="tab:orange", alpha=0.8)
twin.set_ylabel("MFU (%)")
twin.set_ylim(bottom=0)

figure.suptitle(f"nanochat.cpp training metrics on {model.device}",
                fontsize=13)
plot_path = Path("/kaggle/working/training_metrics.png")
figure.savefig(plot_path, dpi=120)
print("saved:", plot_path, f"({plot_path.stat().st_size:,} bytes)")
plt.show()
